In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install -q transformers[sentencepiece]>=4.40.0 datasets>=2.20.0 accelerate>=0.28.0 sentencepiece scikit-learn matplotlib seaborn

In [3]:
import os
os.makedirs("/content/drive/MyDrive/255/outputs", exist_ok=True)
os.makedirs("/content/drive/MyDrive/255/plots", exist_ok=True)
os.makedirs("/content/drive/MyDrive/255/model", exist_ok=True)

In [4]:

!cp /content/drive/MyDrive/255/l3_deberta_finetune.py /content/

!python /content/drive/MyDrive/255/l3_deberta_finetune.py \
  --data_csv /content/drive/MyDrive/255/reviews_enriched.csv \
  --unfreeze_layers 12 \
  --epochs 6 \
  --batch_size 32


Device       : cuda
GPU          : NVIDIA A100-SXM4-80GB
VRAM         : 85.1 GB

Loading data from /content/drive/MyDrive/255/reviews_enriched.csv ...
Dropped 1,272 rows with empty/ultra-short review_text
Total reviews : 529,859
Spam rate     : 13.5%
Unique users  : 239,199

Splitting with StratifiedGroupKFold (groups=user_id) ...
Train : 422,704  (spam: 13.5%)  users: 191,382
Val   : 53,468  (spam: 13.2%)  users: 23,907
Test  : 53,687  (spam: 13.3%)  users: 23,910

Loading tokenizer: microsoft/deberta-base ...
config.json: 100% 474/474 [00:00<00:00, 2.46MB/s]
tokenizer_config.json: 100% 52.0/52.0 [00:00<00:00, 251kB/s]
vocab.json: 899kB [00:00, 102MB/s]
merges.txt: 456kB [00:00, 100MB/s]
Tokenizing datasets ...

Loading model: microsoft/deberta-base ...
pytorch_model.bin: 100% 559M/559M [00:05<00:00, 109MB/s]
Loading weights: 100% 196/196 [00:00<00:00, 990.56it/s, Materializing param=deberta.encoder.rel_embeddings.weight] 
DebertaForSequenceClassification LOAD REPORT from: microsoft/d

In [ ]:
import os

print("Checking output directory:")
!ls -lt /content/drive/MyDrive/255/outputs

print("\nChecking model directory:")
!ls -lt /content/drive/MyDrive/255/model

Checking output directory:
total 15072
-rw------- 1 root root      126 Apr 23 06:13 deberta_threshold_metadata.json
-rw------- 1 root root 15420606 Apr 23 06:13 deberta_predictions.csv
-rw------- 1 root root      550 Apr 23 06:13 deberta_metrics.json
-rw------- 1 root root    10430 Apr 23 06:02 training_history.csv

Checking model directory:
total 368382
-rw------- 1 root root   8339191 Apr 23 06:13 tokenizer.json
-rw------- 1 root root       509 Apr 23 06:13 tokenizer_config.json
-rw------- 1 root root      5201 Apr 23 06:13 training_args.bin
-rw------- 1 root root 368871884 Apr 23 06:13 model.safetensors
-rw------- 1 root root       938 Apr 23 06:13 config.json
drwx------ 2 root root      4096 Apr 23 02:07 checkpoints


In [ ]:
import json
import pandas as pd
import os

metrics_path = '/content/drive/MyDrive/255/outputs/deberta_metrics.json'
history_path = '/content/drive/MyDrive/255/outputs/training_history.csv'

if os.path.exists(metrics_path):
    print("--- Final Training Metrics ---")
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print("Metrics file not found.")

if os.path.exists(history_path):
    print("\n--- Last 5 logs in Training History ---")
    df_history = pd.read_csv(history_path)
    display(df_history.tail())
else:
    print("Training history file not found.")

--- Final Training Metrics ---
{
  "model": "microsoft/deberta-v3-base",
  "unfreeze_layers": 6,
  "trainable_params": 43100930,
  "total_params": 184423682,
  "max_length": 256,
  "epochs_trained": 4,
  "batch_size": 32,
  "grad_accum": 1,
  "effective_batch_size": 32,
  "learning_rate": 2e-05,
  "train_size": 422704,
  "val_size": 53468,
  "test_size": 53687,
  "auc_roc": 0.5013,
  "avg_precision": 0.1335,
  "f1_macro_default": 0.4643,
  "f1_spam_default": 0.0,
  "optimal_threshold": 0.14,
  "f1_macro_optimal": 0.235,
  "f1_spam_optimal": 0.235,
  "runtime_minutes": 64.0
}

--- Last 5 logs in Training History ---


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_f1_macro,eval_f1_spam,eval_auc_roc,eval_avg_precision,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
105,0.403484,0.717773,4.962062e-08,3.898562,51500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,0.391826,1.378906,1.952607e-08,3.936412,52000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
107,0.400668,1.504883,3.211069e-09,3.974262,52500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,NaN,NaN,NaN,4.000000,52840,0.391324,0.464528,0.0,0.5,0.13249,52.9473,1009.834,15.789,NaN,NaN,NaN,NaN,NaN
109,NaN,NaN,NaN,4.000000,52840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3842.6194,440.017,13.751,2.218033e+17,0.39853


In [ ]:
!cat /content/drive/MyDrive/255/outputs/deberta_metrics.json

{
  "model": "microsoft/deberta-v3-base",
  "unfreeze_layers": 6,
  "trainable_params": 43100930,
  "total_params": 184423682,
  "max_length": 256,
  "epochs_trained": 4,
  "batch_size": 32,
  "grad_accum": 1,
  "effective_batch_size": 32,
  "learning_rate": 2e-05,
  "train_size": 422704,
  "val_size": 53468,
  "test_size": 53687,
  "auc_roc": 0.5013,
  "avg_precision": 0.1335,
  "f1_macro_default": 0.4643,
  "f1_spam_default": 0.0,
  "optimal_threshold": 0.14,
  "f1_macro_optimal": 0.235,
  "f1_spam_optimal": 0.235,
  "runtime_minutes": 64.0
}

In [ ]:
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig

model_path = '/content/drive/MyDrive/255/model'

print(f"Checking model files in {model_path}...")
files = os.listdir(model_path)
for f in ['config.json', 'model.safetensors', 'training_args.bin']:
    if f in files:
        size_mb = os.path.getsize(os.path.join(model_path, f)) / (1024*1024)
        print(f"- {f}: Found ({size_mb:.2f} MB)")
    else:
        print(f"- {f}: MISSING")

try:
    print("\nAttempting to load the saved model to verify integrity...")
    config = AutoConfig.from_pretrained(model_path)
    # Loading with device_map='cpu' to avoid OOM just for a check
    model = AutoModelForSequenceClassification.from_pretrained(model_path, config=config, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    print("✅ Model loaded successfully. The save seems complete.")
except Exception as e:
    print(f"❌ Error loading model: {e}")

Checking model files in /content/drive/MyDrive/255/model...
- config.json: Found (0.00 MB)
- model.safetensors: Found (351.78 MB)
- training_args.bin: Found (0.00 MB)

Attempting to load the saved model to verify integrity...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ Model loaded successfully. The save seems complete.


In [ ]:
import os
import re

checkpoint_dir = '/content/drive/MyDrive/255/model/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]

    def get_step(name):
        match = re.search(r'checkpoint-(\d+)', name)
        return int(match.group(1)) if match else 0

    if checkpoints:
        sorted_checkpoints = sorted(checkpoints, key=get_step, reverse=True)
        print(f"Found {len(checkpoints)} checkpoints.")
        print(f"Latest checkpoint (by step number): {sorted_checkpoints[0]}")
        print("\nAll checkpoints found:")
        for ckpt in sorted_checkpoints:
            print(f"- {ckpt}")
    else:
        print("No checkpoint directories found starting with 'checkpoint-'.")
else:
    print(f"Directory not found: {checkpoint_dir}")

Found 2 checkpoints.
Latest checkpoint (by step number): checkpoint-39630

All checkpoints found:
- checkpoint-39630
- checkpoint-26420


In [ ]:
import datetime

def get_mtime(path):
    t = os.path.getmtime(path)
    return datetime.datetime.fromtimestamp(t).strftime('%Y-%m-%d %H:%M:%S')

print("--- Timestamps of main model files ---")
model_path = '/content/drive/MyDrive/255/model'
for f in ['model.safetensors', 'config.json', 'training_args.bin']:
    p = os.path.join(model_path, f)
    if os.path.exists(p):
        print(f"{f}: {get_mtime(p)}")

print("\n--- Timestamps of checkpoints ---")
checkpoint_dir = '/content/drive/MyDrive/255/model/checkpoints'
if os.path.exists(checkpoint_dir):
    for ckpt in sorted_checkpoints:
        p = os.path.join(checkpoint_dir, ckpt)
        print(f"{ckpt}: {get_mtime(p)}")

--- Timestamps of main model files ---
model.safetensors: 2026-04-23 06:13:20
config.json: 2026-04-23 06:13:00
training_args.bin: 2026-04-23 06:13:21

--- Timestamps of checkpoints ---
checkpoint-39630: 2026-04-23 09:55:52
checkpoint-26420: 2026-04-23 08:52:33
